In [1]:
# # AI-Based Cardiovascular Disease Prediction using Hybrid Quantum Neural Networks

# ## Notebook 1 : Dataset Preparation

# ### Objectives

# - Download MIT-BIH Arrhythmia Database
# - Apply ECG preprocessing
# - Segment heartbeats
# - Map heartbeat annotations
# - Create a clean dataset
# - Save dataset for CNN training

# Dataset:
# MIT-BIH Arrhythmia Database

# Output Files:
# - beats.npy
# - labels.npy
# - patient_ids.npy
# - class_mapping.json

In [2]:
!pip -q install wfdb
!pip -q install tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 48.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.


In [3]:
import pandas as pd
import numpy as np
import sklearn
import scipy
import matplotlib

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)

Pandas: 3.0.3
NumPy: 2.0.2
Scikit-learn: 1.6.1
SciPy: 1.16.3
Matplotlib: 3.10.0


In [4]:
import os
import json
import wfdb
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from collections import Counter

from scipy.signal import butter
from scipy.signal import filtfilt

warnings.filterwarnings("ignore")

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [5]:
# ==========================================================
# Mount Google Drive
# ==========================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# ==========================================================
# Project Directory
# ==========================================================

PROJECT_PATH = "/content/drive/MyDrive/Hybrid_QML_CVD_Project"

DATASET_PATH = os.path.join(PROJECT_PATH, "datasets")
MODEL_PATH = os.path.join(PROJECT_PATH, "models")
OUTPUT_PATH = os.path.join(PROJECT_PATH, "outputs")
FIGURE_PATH = os.path.join(PROJECT_PATH, "figures")

os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(FIGURE_PATH, exist_ok=True)

print("Project folders created successfully.")

Project folders created successfully.


In [7]:
# ==========================================================
# Download MIT-BIH Dataset
# ==========================================================

DATABASE_PATH = os.path.join(DATASET_PATH, "mitdb")

if not os.path.exists(DATABASE_PATH):

    print("Downloading MIT-BIH Dataset...")

    wfdb.dl_database(
        "mitdb",
        DATABASE_PATH
    )

    print("Download Completed.")

else:

    print("Dataset already exists.")

Dataset already exists.


In [8]:
# ==========================================================
# ECG Configuration
# ==========================================================

# MIT-BIH Sampling Frequency
SAMPLING_RATE = 360

# Heartbeat window
WINDOW_BEFORE = 100
WINDOW_AFTER = 100
BEAT_LENGTH = WINDOW_BEFORE + WINDOW_AFTER

print("Sampling Rate :", SAMPLING_RATE)
print("Heartbeat Length :", BEAT_LENGTH)

Sampling Rate : 360
Heartbeat Length : 200


In [9]:
# ==========================================================
# Bandpass Filter
# ==========================================================

from scipy.signal import butter, filtfilt

def bandpass_filter(signal,
                    lowcut=0.5,
                    highcut=40,
                    fs=360,
                    order=4):
    """
    Removes baseline wander and high-frequency noise.

    Parameters
    ----------
    signal : numpy array
    ECG signal

    Returns
    -------
    Filtered ECG signal
    """

    nyquist = 0.5 * fs

    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(order,
                  [low, high],
                  btype='band')

    filtered = filtfilt(b, a, signal)

    return filtered

In [10]:
# ==========================================================
# Heartbeat Segmentation
# ==========================================================

def segment_heartbeat(signal,
                      r_peak,
                      before=100,
                      after=100):
    """
    Extract heartbeat centered around R peak.

    Returns
    -------
    heartbeat (200 samples)

    or

    None if window exceeds signal.
    """

    start = r_peak - before
    end = r_peak + after

    if start < 0:
        return None

    if end > len(signal):
        return None

    heartbeat = signal[start:end]

    return heartbeat

In [11]:
# ==========================================================
# Heartbeat Normalization
# ==========================================================

def normalize_heartbeat(heartbeat):
    """
    Z-score normalization.

    Each heartbeat is normalized independently.
    """

    mean = np.mean(heartbeat)
    std = np.std(heartbeat)

    if std == 0:
        return heartbeat

    return (heartbeat - mean) / std

In [12]:
# ==========================================================
# MIT-BIH Label Mapping
# ==========================================================

LABEL_MAP = {

    # Normal Beats
    'N': 'Normal',
    'L': 'LBBB',
    'R': 'RBBB',

    # Atrial
    'A': 'APB',
    'a': 'APB',
    'J': 'APB',
    'S': 'APB',

    # Ventricular
    'V': 'PVC',
    'E': 'PVC'
}

print("Total Supported Labels :", len(LABEL_MAP))

Total Supported Labels : 9


In [13]:
# ==========================================================
# Save Class Mapping
# ==========================================================

mapping_path = os.path.join(DATASET_PATH,
                            "class_mapping.json")

with open(mapping_path, "w") as f:
    json.dump(LABEL_MAP,
              f,
              indent=4)

print("Class mapping saved successfully.")

Class mapping saved successfully.


In [14]:
# ==========================================================
# Test Helper Functions
# ==========================================================

record = wfdb.rdrecord(os.path.join(DATABASE_PATH, "100"))
annotation = wfdb.rdann(os.path.join(DATABASE_PATH, "100"), "atr")

# Select MLII Lead
signal = record.p_signal[:, 0]

# Apply bandpass filter
filtered = bandpass_filter(signal)

# Choose a valid R-peak (skip the first few)
r_peak = annotation.sample[3]   # 662

print("Selected R-Peak:", r_peak)

# Extract heartbeat
heartbeat = segment_heartbeat(filtered, r_peak)

# Check before normalization
if heartbeat is None:
    print("Heartbeat extraction failed!")
else:
    heartbeat = normalize_heartbeat(heartbeat)

    print("Signal Length :", len(signal))
    print("Heartbeat Length :", len(heartbeat))
    print("Heartbeat Mean :", np.mean(heartbeat))
    print("Heartbeat Std :", np.std(heartbeat))

Selected R-Peak: 662
Signal Length : 650000
Heartbeat Length : 200
Heartbeat Mean : -8.326672684688674e-18
Heartbeat Std : 1.0


In [15]:
print("First 10 annotation samples:")
print(annotation.sample[:10])

r_peak = annotation.sample[1]
print("Selected R Peak:", r_peak)

heartbeat = segment_heartbeat(filtered, r_peak)

print("Heartbeat is None?", heartbeat is None)

if heartbeat is not None:
    print("Heartbeat Shape:", heartbeat.shape)

First 10 annotation samples:
[  18   77  370  662  946 1231 1515 1809 2044 2402]
Selected R Peak: 77
Heartbeat is None? True


In [16]:
# ==========================================================
# MIT-BIH Records to Process
# ==========================================================

RECORDS = [
    '100','101','103','105','106','107','108','109',
    '111','112','113','114','115','116','117','118',
    '119','121','122','123','124','200','201','202',
    '203','205','207','208','209','210','212','213',
    '214','215','217','219','220','221','222','223',
    '228','230','231','232','233','234'
]

print("Total Records:", len(RECORDS))

Total Records: 46


In [17]:
# ==========================================================
# Storage Lists
# ==========================================================

beats = []
labels = []
patient_ids = []

print("Storage initialized.")

Storage initialized.


In [18]:
# ==========================================================
# Process All MIT-BIH Records
# ==========================================================

# Clear storage lists in case this cell is re-run
beats = []
labels = []
patient_ids = []

print("Processing MIT-BIH records...\n")

for record_name in tqdm(RECORDS):

    try:
        # --------------------------------------------------
        # Read ECG Record and Annotation
        # --------------------------------------------------
        record = wfdb.rdrecord(os.path.join(DATABASE_PATH, record_name))
        annotation = wfdb.rdann(os.path.join(DATABASE_PATH, record_name), "atr")

    except Exception as e:
        print(f"Skipping Record {record_name}: {e}")
        continue

    # ------------------------------------------------------
    # Select MLII Lead
    # ------------------------------------------------------
    signal = record.p_signal[:, 0]

    # ------------------------------------------------------
    # Apply Bandpass Filter
    # ------------------------------------------------------
    signal = bandpass_filter(signal)

    # ------------------------------------------------------
    # Process Every Annotated Beat
    # ------------------------------------------------------
    for r_peak, symbol in zip(annotation.sample, annotation.symbol):

        # Ignore unsupported beat types
        if symbol not in LABEL_MAP:
            continue

        # Segment heartbeat
        heartbeat = segment_heartbeat(
            signal,
            r_peak,
            before=WINDOW_BEFORE,
            after=WINDOW_AFTER
        )

        # Skip invalid heartbeat windows
        if heartbeat is None:
            continue

        # Normalize heartbeat
        heartbeat = normalize_heartbeat(heartbeat)

        # Final safety check
        if len(heartbeat) != BEAT_LENGTH:
            continue

        # Save heartbeat
        beats.append(heartbeat.astype(np.float32))
        labels.append(LABEL_MAP[symbol])
        patient_ids.append(int(record_name))

print("\nProcessing Completed Successfully!")

print(f"Total Heartbeats Collected : {len(beats)}")

Processing MIT-BIH records...



100%|██████████| 46/46 [00:49<00:00,  1.08s/it]


Processing Completed Successfully!
Total Heartbeats Collected : 100106


In [19]:
# ==========================================================
# Convert to NumPy Arrays
# ==========================================================

beats = np.array(beats, dtype=np.float32)
labels = np.array(labels)
patient_ids = np.array(patient_ids, dtype=np.int32)

print("Beats Shape      :", beats.shape)
print("Labels Shape     :", labels.shape)
print("Patients Shape   :", patient_ids.shape)

Beats Shape      : (100106, 200)
Labels Shape     : (100106,)
Patients Shape   : (100106,)


In [20]:
# ==========================================================
# Dataset Statistics
# ==========================================================

from collections import Counter

distribution = Counter(labels)

print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)

print(f"Total Heartbeats : {len(beats)}")
print(f"Heartbeat Length : {beats.shape[1]}")
print(f"Unique Patients  : {len(np.unique(patient_ids))}")

print("\nClass Distribution\n")

for cls in sorted(distribution.keys()):
    print(f"{cls:10s} : {distribution[cls]}")

print("=" * 50)

DATASET SUMMARY
Total Heartbeats : 100106
Heartbeat Length : 200
Unique Patients  : 46

Class Distribution

APB        : 2781
LBBB       : 8073
Normal     : 74766
PVC        : 7229
RBBB       : 7257


In [21]:
# ==========================================================
# Save Processed Dataset
# ==========================================================

np.save(os.path.join(DATASET_PATH, "beats.npy"), beats)
np.save(os.path.join(DATASET_PATH, "labels.npy"), labels)
np.save(os.path.join(DATASET_PATH, "patient_ids.npy"), patient_ids)

print("Dataset saved successfully!")

Dataset saved successfully!


In [22]:
# ==========================================================
# Save Label Mapping
# ==========================================================

CLASS_TO_INDEX = {
    "Normal": 0,
    "LBBB": 1,
    "RBBB": 2,
    "APB": 3,
    "PVC": 4
}

INDEX_TO_CLASS = {v: k for k, v in CLASS_TO_INDEX.items()}

with open(os.path.join(DATASET_PATH, "class_mapping.json"), "w") as f:
    json.dump(CLASS_TO_INDEX, f, indent=4)

print(CLASS_TO_INDEX)

{'Normal': 0, 'LBBB': 1, 'RBBB': 2, 'APB': 3, 'PVC': 4}


In [23]:
# ==========================================================
# Verify Saved Files
# ==========================================================

files = os.listdir(DATASET_PATH)

print("Files in dataset folder:\n")

for file in sorted(files):
    print(file)

Files in dataset folder:

beats.npy
class_mapping.json
labels.npy
mitdb
patient_ids.npy


In [24]:
import os

files = sorted(os.listdir(DATABASE_PATH))

records = sorted(set([f.split(".")[0] for f in files]))

print("Number of records found:", len(records))
print(records)

Number of records found: 48
['100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '111', '112', '113', '114', '115', '116', '117', '118', '119', '121', '122', '123', '124', '200', '201', '202', '203', '205', '207', '208', '209', '210', '212', '213', '214', '215', '217', '219', '220', '221', '222', '223', '228', '230', '231', '232', '233', '234']


In [25]:
import shutil
import os

if os.path.exists(DATABASE_PATH):
    shutil.rmtree(DATABASE_PATH)
    print("Old dataset deleted.")

Old dataset deleted.


In [26]:
DATABASE_PATH = os.path.join(DATASET_PATH, "mitdb")

print("Downloading MIT-BIH dataset...")

wfdb.dl_database("mitdb", DATABASE_PATH)

print("Download completed!")

Generating record list for: 100
Generating record list for: 101
Generating record list for: 102
Generating record list for: 103
Generating record list for: 104
Generating record list for: 105
Generating record list for: 106
Generating record list for: 107
Generating record list for: 108
Generating record list for: 109
Generating record list for: 111
Generating record list for: 112
Generating record list for: 113
Generating record list for: 114
Generating record list for: 115
Generating record list for: 116
Generating record list for: 117
Generating record list for: 118
Generating record list for: 119
Generating record list for: 121
Generating record list for: 122
Generating record list for: 123
Generating record list for: 124
Generating record list for: 200
Generating record list for: 201
Generating record list for: 202
Generating record list for: 203
Generating record list for: 205
Generating record list for: 207
Generating record list for: 208
Generating record list for: 209
Generati

In [27]:
files = sorted(os.listdir(DATABASE_PATH))
records = sorted(set(f.split(".")[0] for f in files if f.endswith(".dat")))

print("Total records:", len(records))
print(records[-5:])

Total records: 48
['230', '231', '232', '233', '234']
